%md
## 🧪 Evaluating the Production Agent

In the previous section, we built and deployed a **production-grade Financial Intelligence Agent** using LangGraph.

This agent is capable of:

- 📊 Querying structured financial data (Genie)  
- 📄 Understanding unstructured reports (Knowledge Assistant)  
- 🧠 Making dynamic decisions through a custom Supervisor  
- 🔗 Orchestrating everything via a LangGraph workflow  
- 🚀 Running as a real-time API endpoint  

---

#### ❓ But now comes the most important question:

> **How good is it, really?**

---

### 🧠 Why Evaluation Matters

Building an agent is only half the job.

In production, we need to ensure:

- ✅ **Accuracy** → Are the answers correct?  
- 🎯 **Relevance** → Does it answer the actual question?  
- 🛡️ **Safety** → Is the response appropriate and reliable?  

---

💡 Just like in Part 1, we will use:

> **LLM-as-a-Judge via MLflow GenAI**

### 📦 Setup

We install the required dependencies.

> 💡 This step ensures a consistent and reproducible environment for evaluation.

In [0]:
%pip install databricks-openai mlflow==3.6.0
%restart_python

In [0]:
import mlflow
import mlflow.deployments
import pandas as pd
from databricks.sdk import WorkspaceClient
from mlflow.genai import evaluate
from mlflow.genai.scorers import Correctness, RelevanceToQuery, Safety

### ⚙️ Configuration

We define the key components of our evaluation setup:

- **Endpoint** → The deployed LangGraph Agent  
- **Judge Model** → An LLM used to score responses  
- **MLflow Experiment** → Where results are tracked  

In [0]:
### ✍️ ADD YOUR ENDPOINT HERE ###
ENDPOINT_NAME = ""
JUDGE_MODEL = "databricks-meta-llama-3-3-70b-instruct"
EXPERIMENT_PATH = f"/Users/{dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()}/agent_evaluation"

# Initialize Workspace Client
w = WorkspaceClient()

# Set the experiment
mlflow.set_experiment(EXPERIMENT_PATH)

### 🎯 Evaluation Dataset


In [0]:
# Create a small test set (Query + Reference Answer)
eval_data = pd.DataFrame([
    {
        "query": "What were Avolta's reported net sales and turnover for the six months ended June 30, 2025?",
        "expected_response": "Avolta reported net sales of CHF 6,624 million and a total turnover of CHF 6,734 million for the six months ended June 30, 2025."
    },
    {
        "query": "How many treasury shares did Avolta cancel in December 2024, and what was their par value?",
        "expected_response": "In December 2024, Avolta cancelled 6,104 thousand shares, which had a par value of CHF 5.00 each."
    },
    {
        "query": "What is Avolta's medium-term leverage target?",
        "expected_response": "Avolta's medium-term leverage target is 1.5x to 2.0x net debt/CORE EBITDA, with the flexibility to go up to 2.5x."
    },
    {
        "query": "What was the CORE EBITDA for the year ended December 31, 2024?",
        "expected_response": "The CORE EBITDA for the year ended December 31, 2024, was CHF 1,267 million."
    },
    {
        "query": "How much was paid in dividends to Group shareholders during the first six months of 2025?",
        "expected_response": "During the first six months of 2025, CHF 143 million was paid in dividends to Group shareholders"
    }
])

# Format for MLflow Evaluation
eval_dataset = pd.DataFrame({
    "inputs": [{"query": q} for q in eval_data["query"]],
    "expectations": [{"expected_response": a} for a in eval_data["expected_response"]]
})

### 🔌 Prediction Function

This function connects MLflow to our **live deployed agent**.

---

### 🔄 What happens here?

1. Send a query to the endpoint  
2. The full LangGraph pipeline runs (Normalizer → Genie → Supervisor → etc.)  
3. Extract the **final generated answer**  

In [0]:
def predict_fn(query):
    client = mlflow.deployments.get_deploy_client("databricks")
    
    try:
        # Construct the exact payload the Agent requires
        payload = {
            "input": [{"role": "user", "content": query}]
        }
        
        # Query the endpoint directly
        response = client.predict(endpoint=ENDPOINT_NAME, inputs=payload)
        
        # Navigate the multi-turn response to find the final text
        return response["output"][-1]["content"][0]["text"]
        
    except Exception as e:
        # If an error occurs make sure the whole evaluation doesn't break
        return f"ERROR: {str(e)}"

### 📊 Running the Evaluation

**Next Step:**  
Open the MLflow experiment to explore results and compare performance.

In [0]:
# Define the Scorers
correctness_metric = Correctness(model=f"endpoints:/{JUDGE_MODEL}")
relevance_metric = RelevanceToQuery(model=f"endpoints:/{JUDGE_MODEL}")
safety_metric = Safety(model=f"endpoints:/{JUDGE_MODEL}")

# Run the Evaluation
run_name=f"langgraph_eval_{pd.Timestamp.now().strftime('%H%M%S')}"
with mlflow.start_run(run_name=run_name):
    results = evaluate(
        data=eval_dataset,
        predict_fn=predict_fn,
        scorers=[correctness_metric, relevance_metric, safety_metric],
    )

print("✅ Evaluation Complete!")

## 🎉 End of Workshop — From No-Code to Production AI

Let’s take a step back and look at what we've seen.

---

### 🧱 Part 1 — No-Code Agent (Agent Bricks)

- 📄 Indexed documents with RAG  
- 📊 Queried structured data with Genie  
- 🚦 Built a Supervisor agent  
- 🧪 Evaluated performance  

---

### ⚙️ Part 2 — Custom Agent Engineering (LangGraph)

- 🧠 Designed a full agent architecture  
- 🧩 Built each node with custom logic  
- 🔗 Orchestrated dynamic workflows  
- 📦 Wrapped and deployed with MLflow  
- 🚀 Served as a production endpoint  
- 🧪 Evaluated with the same framework

---

### 🤯 What’s the Key Takeaway?

You now understand both worlds:

| Feature | 🟢 No-Code (Agent Bricks) | 🔵 Code-Based (LangGraph) |
|--------|---------------------------|---------------------------|
| Speed to Market | Instant. Protyping is easy. | Slower. Requires development and testing cycles. |
| Customization | Limited. You play within the tools provided. | Infinite. If you can code the logic, the agent can do it. |
| Governance | Built-in. Security and tracking are standard. | Bespoke. You must design how it handles data/security. |
| Maintenance | Low. Databricks manages the underlying updates. | High. You own the "code debt" and dependency versions. |
| Complexity | Best for standard RAG and simple tasks. | Best for complex, multi-step workflows and unique logic. |
---

### 🧠 Why This Matters

In real projects, you will often:

- Start with **no-code tools** for speed ⚡  
- Move to **custom architectures** for control 🧠  
- Use **evaluation frameworks** to improve 📊  
- Deploy as **scalable endpoints** 🚀  

---

### 🚀 Final Thought

You didn’t just build a chatbot today. 

👏 You built a complete AI system lifecycle.